In [1]:
import time, json, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import pyvisa
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [2]:
from K2450 import K2450
import ABAA
from ABAA import (BASE, CAL, measure_R, breakdown_ramp, tune, hold,
                  tune_and_hold, fit_active)

In [3]:
# ---- edit these ----
RESOURCE  = "TCPIP0::192.168.50.2::inst0::INSTR"   # pyvisa.ResourceManager().list_resources()
DATA_DIR  = Path(r'E:\ABAA\Data')
SAMPLE    = "witness_batch_01"
ROOM_T_C  = None          # chuck / room temperature in C, by hand
FOUR_WIRE = False         # True: two needles per pad, sense leads on the second pair

In [4]:
# burst (Križan defaults)
M_PULSES  = 6             # bipolar pulses per iteration
T_PULSE   = 0.200         # s at +V_A and at -V_A (one 200 ms bias cycle, Križan Fig. 1a)
T_BLANK   = 0.100         # s at 0 V after each polarity (100 ms blanking)
DT_POINT  = 0.025         # s per list point; T_PULSE and T_BLANK must be multiples of it
NPLC_BURST = 0.05         # fast current reading inside the burst (protection only)

In [5]:
# I-V readout
V_IV      = 0.013         # sweep -V_IV .. +V_IV
N_IV      = 27            # points (1 mV steps)
NPLC_IV   = 1.0
DELAY_IV  = 0.005

R_SHORT   = 260.0         # Ohm: below this a junction counts as shorted (Križan)

DATA_DIR.mkdir(exist_ok=True)

In [6]:
smu = K2450(RESOURCE)
smu.configure(ilim=1e-3)
smu.set_v(0.0)

KEITHLEY INSTRUMENTS,MODEL 2450,MYFP002618,1.7.16a


In [7]:
print(smu.q(":ROUT:TERM?"))

REAR


In [ ]:
# try:
#     smu.close()
# except Exception:
#     pass

In [ ]:
ABAA.setup(smu, sample=SAMPLE, data_dir=DATA_DIR, room_T_C=ROOM_T_C, r_short=R_SHORT)

## Dummy-resistor check (5 kΩ on the probes — **not a junction**)
Expect R ~ 10 kΩ, both before and after the bursts, and measured pulse/blank widths close to the requested ones. If the timing is off, adjust `DT_POINT` or `NPLC_BURST`. Watch for `SMU errors:`.

In [ ]:
smu.timing_reported = False
measure_R("DUMMY")
smu.set_ilim(1e-3)
b = smu.burst(0.5, M=2)
print(b)
measure_R("DUMMY")
smu.check_errors()

In [ ]:
smu.output(True)
smu.burst(0.3, M=1)
print("Output:", smu.q(":OUTP?"), " Set V:", smu.q(":SOUR:VOLT?"))
smu.output(False)

In [ ]:
smu.timing_reported = False
b = smu.burst(0.1)      # full 6-pulse burst, 144 points
print(b)
print("Errors:", smu.check_errors())

In [ ]:
smu.iv()

In [ ]:
smu.burst(0.2, M=3)

In [ ]:
smu.output(False)